## **1. Préliminaires**

### **1.1. Contexte**
DATA/projects/heart_attack_prediction/project/notebooks/analysis.ipynb

### **1.2. Imports des librairies**

Dans cette section nous allons importer les librairies principales .

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### **1.3. Chargement des données**
Nous allons maintenant charger notre dataset .

In [ ]:
# chargement du dataset
data = pd.read_csv('../data/data-cleaned.csv')

# affichage d'un extrait
data.head()

### **1.4. Structure et typologie des données**
Nous allons analyser la structure de notre jeu de données et le types des données et changer les types de données si nécessaires.

In [ ]:
# dimension du dataset
data.shape

In [ ]:
# affichage des infos sur les variables et leurs types
data.info()

## **2. Analyse Exploratoire des données**

### **2.1. Analyse univariée**

Nous allons observer la distribution de chaque variable.
Nous allons pour certains graphiques utiliser des diagramme à baton et pour d'autres des diagrammes circulaires.

### **2.2. Analyse Multivariée**

 Matrice des corrélations

In [ ]:
# tableau des corrélations
tab_corr = data.corr()
tab_corr

In [ ]:
# matrice des corrélations

plt.figure(figsize=(20,10))
sns.heatmap(vmin=-1,vmax=1,data=tab_corr,cmap="coolwarm",annot=True,fmt=".2f",linewidths=1.0)

plt.show()

### **2.3** Analyse par composante principales

#### **2.3.1** Preparation des données

In [ ]:
numerical_cols = []
X = data[numerical_cols].copy()

#### **2.3.2** Standarisation
Nous allons normaliser les données

In [ ]:
from sklearn.preprocessing import StandardScaler

# initialisation du scaler
scaler = StandardScaler()

# matrice centrée et réduite
X_Scaled = scaler.fit_transform(X)

# verification
pd.DataFrame(X_Scaled).describe().round(2).loc[['std','mean']]

#### **2.3.3** ACP

In [ ]:
from sklearn.decomposition import PCA

# nombre de composantes
n_components  = 3

# instanciation du PCA
pca = PCA(n_components=n_components)

# on entraine notre PCA
pca.fit(X_Scaled)

#### **2.3.4** Variance expliquée et scree plot

In [ ]:
for i , v in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1} : {v:.2%} de variance expliquée")

In [ ]:
# enregistrons les variances expliquées
scree = (pca.explained_variance_ratio_*100).round(2)
scree

In [ ]:
# variances expliquées cumulées
scree_cum = np.cumsum(scree).round()
scree_cum

In [ ]:
# labels des composantes
components_labels = [str(i+1) for i in range(n_components)]
components_labels

In [ ]:
# graphique de l'eboulis des valeurs propres

plt.figure(figsize=(12,9))

sns.barplot(x=components_labels,y=scree,errorbar=None)

plt.plot(components_labels,scree_cum, c="red",marker="o")

plt.yticks(ticks=np.arange(0,110,10))

plt.xlabel("rang des composantes")
plt.ylabel("pourcentage d'inertie")
plt.title("Eboulis des valeurs propres")

plt.show()

#### **2.3.5** `Les composantes`

In [ ]:
# transformation de la matrice des composantes en dataframe
pcs = pd.DataFrame(pca.components_)
pcs.head()

In [ ]:
pcs.columns = X.columns
pcs.index = [f"F{i}" for i in components_labels]
pcs.head()

#### **2.3.6** `Matrice des corrélations`

In [ ]:
# matrice de corrélation

plt.figure(figsize=(15,10))

sns.heatmap(vmin=-1,vmax=1,annot=True,linewidths=2,data=pcs.T,cmap="coolwarm")
plt.show()

#### **2.3.7** `Graphe des corrélations`

In [ ]:
def correlation_graph(x_y,pca:PCA,features:list):
    # initialisation des axes
    x,y=x_y

    # initialisation de la figure
    fig,ax = plt.subplots(figsize=(12,9))

    # tracée des flêches
    for i in range(len(features)):
        plt.arrow(
            x=0,
            y=0,
            dx=pca.components_[x,i],
            dy=pca.components_[y,i],
            head_width=0.04,
            head_length=0.04,
        )

        plt.text(
            x=pca.components_[x,i]+0.05,
            y=pca.components_[y,i]+0.05,
            s=features[i]
            )

    # tracée des axes
    plt.plot([-1,1],[0,0],color="grey",alpha=0.8,ls="--")
    plt.plot([0, 0],[-1, 1], color="grey", alpha=0.8, ls="--")

    # nommage des axes
    v1,v2 = [f"{pca.explained_variance_ratio_[k]:.1%}" for k in x_y]
    plt.xlabel(f"F{x+1} {v1}")
    plt.ylabel(f"F{y+1} {v2}")

    # tracée du cercle
    theta = np.linspace(0,2*np.pi,100)
    plt.plot(np.cos(theta),np.sin(theta))
    plt.axis("equal")

    # titre
    plt.title(f"Cercle des corrélations (F{x+1} et F{y+1})")

    # affichage
    plt.show()

In [ ]:
# graphe F1 et F2
x_y = 0,1
correlation_graph(x_y=x_y, pca=pca, features=numerical_cols)

In [ ]:
# graphe F3 et F4
x_y = 2, 3
correlation_graph(x_y=x_y, pca=pca, features=numerical_cols)

#### **2.3.8** `Projection sur les plans factoriels`

In [ ]:
X_Projected = pca.transform(X_Scaled)

In [ ]:
def display_factorial_planes(
    x_y,
    X_projected,
    pca=None,
    labels=None,
    marker=".",
    figsize=(12, 9),
    clusters=None,
    alpha=1,
):

    # transformation en array numpy
    X_ = np.array(X_projected)

    # verification des input
    labels = [] if labels is None else labels

    if not len(x_y) == 2 or max(x_y) >= X_.shape[1]:
        raise AttributeError("2 axes sont requis")

    # initialisation des axes
    x, y = x_y

    # initialisation de la figure
    fig,ax = plt.subplots(figsize=figsize)

    # dessin des points
    sns.scatterplot(x=X_[:,x],y=X_[:,y],hue=clusters)

    # tracée des limites des axes
    x_max = np.abs(X_[:,x]).max()*1.1
    y_max = np.abs(X_[:,y]).max()*1.1

    plt.xlim(left=-x_max,right=y_max)
    plt.ylim(bottom=-y_max,top=y_max)

    # tracée des axes
    plt.plot([-x_max,x_max],[0,0],color="grey",alpha=alpha,ls="--")
    plt.plot([0, 0], [-y_max, y_max], color="grey", alpha=alpha, ls="--")

    # nommage des axes
    v1,v2 = ['',''] if pca is None else [f"{pca.explained_variance_ratio_[k]:.1%}" for k in x_y]

    plt.xlabel(f"F {x+1} ({v1})")
    plt.ylabel(f"F{y+1} {v2}")

    # titre et affichage
    plt.title(f"Projection sur F{x+1} et F{y+1}")
    plt.show()

In [ ]:
# projections sur F1 et F2
x_y=(0,1)
display_factorial_planes(x_y=x_y,X_projected=X_Projected,pca=pca)

In [ ]:
# projections sur F3 et F4
x_y = (2, 3)
display_factorial_planes(x_y=x_y, X_projected=X_Projected, pca=pca)